In [37]:
import pandas as pd

df = pd.read_csv("bank_transactions_data_2.csv")

print(df.head())

  TransactionID AccountID  TransactionAmount      TransactionDate  \
0      TX000001   AC00128              14.09  2023-04-11 16:29:14   
1      TX000002   AC00455             376.24  2023-06-27 16:44:19   
2      TX000003   AC00019             126.29  2023-07-10 18:16:08   
3      TX000004   AC00070             184.50  2023-05-05 16:32:11   
4      TX000005   AC00411              13.45  2023-10-16 17:51:24   

  TransactionType   Location DeviceID      IP Address MerchantID Channel  \
0           Debit  San Diego  D000380  162.198.218.92       M015     ATM   
1           Debit    Houston  D000051     13.149.61.4       M052     ATM   
2           Debit       Mesa  D000235  215.97.143.157       M009  Online   
3           Debit    Raleigh  D000187  200.13.225.150       M002  Online   
4          Credit    Atlanta  D000308    65.164.3.100       M091  Online   

   CustomerAge CustomerOccupation  TransactionDuration  LoginAttempts  \
0           70             Doctor                   81 

In [38]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2512 entries, 0 to 2511
Data columns (total 16 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   TransactionID            2512 non-null   object 
 1   AccountID                2512 non-null   object 
 2   TransactionAmount        2512 non-null   float64
 3   TransactionDate          2512 non-null   object 
 4   TransactionType          2512 non-null   object 
 5   Location                 2512 non-null   object 
 6   DeviceID                 2512 non-null   object 
 7   IP Address               2512 non-null   object 
 8   MerchantID               2512 non-null   object 
 9   Channel                  2512 non-null   object 
 10  CustomerAge              2512 non-null   int64  
 11  CustomerOccupation       2512 non-null   object 
 12  TransactionDuration      2512 non-null   int64  
 13  LoginAttempts            2512 non-null   int64  
 14  AccountBalance          

In [39]:
df['TransactionDate'] = pd.to_datetime(df['TransactionDate'])
df['PreviousTransactionDate'] = pd.to_datetime(df['PreviousTransactionDate'])

print(df.dtypes)

TransactionID                      object
AccountID                          object
TransactionAmount                 float64
TransactionDate            datetime64[ns]
TransactionType                    object
Location                           object
DeviceID                           object
IP Address                         object
MerchantID                         object
Channel                            object
CustomerAge                         int64
CustomerOccupation                 object
TransactionDuration                 int64
LoginAttempts                       int64
AccountBalance                    float64
PreviousTransactionDate    datetime64[ns]
dtype: object


In [40]:
df['time_since_last_txn'] = (
    df['TransactionDate'] - df['PreviousTransactionDate']
).dt.total_seconds()

print(df[['TransactionDate', 'PreviousTransactionDate', 'time_since_last_txn']].head())

      TransactionDate PreviousTransactionDate  time_since_last_txn
0 2023-04-11 16:29:14     2024-11-04 08:08:08          -49477134.0
1 2023-06-27 16:44:19     2024-11-04 08:09:35          -42823516.0
2 2023-07-10 18:16:08     2024-11-04 08:07:04          -41694656.0
3 2023-05-05 16:32:11     2024-11-04 08:09:06          -47403415.0
4 2023-10-16 17:51:24     2024-11-04 08:06:39          -33228915.0


In [41]:
df = df.sort_values(by=['AccountID', 'TransactionDate'])

df['prev_txn_date_fixed'] = df.groupby('AccountID')['TransactionDate'].shift(1)

df['time_since_last_txn_fixed'] = (
    df['TransactionDate'] - df['prev_txn_date_fixed']
).dt.total_seconds()

print(df[['AccountID', 'TransactionDate', 'prev_txn_date_fixed', 'time_since_last_txn_fixed']].head(10))

     AccountID     TransactionDate prev_txn_date_fixed  \
1312   AC00001 2023-09-15 17:00:20                 NaT   
2016   AC00001 2023-11-14 16:56:34 2023-09-15 17:00:20   
2120   AC00002 2023-01-10 16:00:32                 NaT   
20     AC00002 2023-02-28 16:36:58 2023-01-10 16:00:32   
1476   AC00002 2023-05-05 16:35:44 2023-02-28 16:36:58   
61     AC00002 2023-05-16 16:07:30 2023-05-05 16:35:44   
1598   AC00002 2023-07-24 16:14:05 2023-05-16 16:07:30   
1673   AC00002 2023-09-11 17:52:59 2023-07-24 16:14:05   
1028   AC00002 2023-12-21 17:00:50 2023-09-11 17:52:59   
2325   AC00003 2023-01-02 16:45:05                 NaT   

      time_since_last_txn_fixed  
1312                        NaN  
2016                  5183774.0  
2120                        NaN  
20                    4235786.0  
1476                  5702326.0  
61                     948706.0  
1598                  5961995.0  
1673                  4239534.0  
1028                  8723271.0  
2325                 

In [42]:
df['high_velocity_flag'] = df['time_since_last_txn_fixed'] < 60

print(df['high_velocity_flag'].value_counts())

high_velocity_flag
False    2511
True        1
Name: count, dtype: int64


In [43]:
df['high_velocity_flag'] = df['time_since_last_txn_fixed'] < 300  # 5 minutes

print(df['high_velocity_flag'].value_counts())

high_velocity_flag
False    2511
True        1
Name: count, dtype: int64


In [44]:
df['avg_txn_amount_per_user'] = df.groupby('AccountID')['TransactionAmount'].transform('mean')

print(df[['AccountID', 'TransactionAmount', 'avg_txn_amount_per_user']].head())

     AccountID  TransactionAmount  avg_txn_amount_per_user
1312   AC00001              47.79               130.380000
2016   AC00001             212.97               130.380000
2120   AC00002             476.99               293.744286
20     AC00002              59.32               293.744286
1476   AC00002              12.62               293.744286


In [45]:
df['amount_deviation_abs'] = abs((
    df['TransactionAmount'] - df['avg_txn_amount_per_user']
) / df['avg_txn_amount_per_user'])

print(df[['TransactionAmount', 'avg_txn_amount_per_user', 'amount_deviation_abs']].head())

      TransactionAmount  avg_txn_amount_per_user  amount_deviation_abs
1312              47.79               130.380000              0.633456
2016             212.97               130.380000              0.633456
2120             476.99               293.744286              0.623827
20                59.32               293.744286              0.798056
1476              12.62               293.744286              0.957037


In [46]:
df['amount_anomaly_flag'] = df['amount_deviation_abs'] > 1

print(df['amount_anomaly_flag'].value_counts())

amount_anomaly_flag
False    2183
True      329
Name: count, dtype: int64


In [47]:
df['location_count_per_user'] = df.groupby('AccountID')['Location'].transform('nunique')

print(df[['AccountID', 'Location', 'location_count_per_user']].head(10))

     AccountID     Location  location_count_per_user
1312   AC00001       Denver                        2
2016   AC00001      Atlanta                        2
2120   AC00002    San Diego                        7
20     AC00002  Los Angeles                        7
1476   AC00002      El Paso                        7
61     AC00002       Dallas                        7
1598   AC00002    Milwaukee                        7
1673   AC00002    Las Vegas                        7
1028   AC00002    Charlotte                        7
2325   AC00003  San Antonio                        5


In [48]:
def location_risk(count):
    if count <= 3:
        return "Low"
    elif count <= 6:
        return "Medium"
    else:
        return "High"

df['location_risk_level'] = df['location_count_per_user'].apply(location_risk)

print(df['location_risk_level'].value_counts())

location_risk_level
Medium    1261
High       885
Low        366
Name: count, dtype: int64


In [49]:
df['fraud_score'] = (
    df['amount_anomaly_flag'].astype(int) * 2 +   # strong signal
    df['high_velocity_flag'].astype(int) * 1 +    # weaker signal
    (df['location_risk_level'] == 'High').astype(int) * 2
)

print(df['fraud_score'].value_counts())

fraud_score
0    1416
2     976
4     119
1       1
Name: count, dtype: int64


In [50]:
def classify_fraud(score):
    if score >= 4:
        return "High Fraud Risk"
    elif score >= 2:
        return "Medium Risk"
    else:
        return "Low Risk"

df['fraud_label'] = df['fraud_score'].apply(classify_fraud)

print(df['fraud_label'].value_counts())

fraud_label
Low Risk           1417
Medium Risk         976
High Fraud Risk     119
Name: count, dtype: int64


In [51]:
# % of high fraud transactions
high_fraud_percentage = (df['fraud_label'] == 'High Fraud Risk').mean() * 100

print("High Fraud %:", high_fraud_percentage)

# Average amount for fraud vs normal
print(df.groupby('fraud_label')['TransactionAmount'].mean())

# Avg login attempts
print(df.groupby('fraud_label')['LoginAttempts'].mean())

High Fraud %: 4.737261146496816
fraud_label
High Fraud Risk    801.138824
Low Risk           225.473917
Medium Risk        340.905236
Name: TransactionAmount, dtype: float64
fraud_label
High Fraud Risk    1.075630
Low Risk           1.125618
Medium Risk        1.129098
Name: LoginAttempts, dtype: float64


In [52]:
from sklearn.ensemble import IsolationForest

features = [
    'TransactionAmount',
    'amount_deviation_abs',
    'time_since_last_txn_fixed',
    'location_count_per_user'
]

X = df[features].fillna(0)

model = IsolationForest(contamination=0.05, random_state=42)
df['anomaly_score'] = model.fit_predict(X)

print(df['anomaly_score'].value_counts())

anomaly_score
 1    2386
-1     126
Name: count, dtype: int64


In [53]:
df['ml_flag'] = df['anomaly_score'] == -1

comparison = pd.crosstab(df['fraud_label'], df['ml_flag'])

print(comparison)

ml_flag          False  True 
fraud_label                  
High Fraud Risk     78     41
Low Risk          1380     37
Medium Risk        928     48


In [56]:
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="WJ28@krhps",  # <-- put real password
    database="fraud_db"
)

cursor = conn.cursor()

print("Connected successfully")

Connected successfully


In [57]:
create_table_query = """
CREATE TABLE IF NOT EXISTS transactions (
    TransactionID VARCHAR(20),
    AccountID VARCHAR(20),
    TransactionAmount FLOAT,
    TransactionDate DATETIME,
    TransactionType VARCHAR(10),
    Location VARCHAR(50),
    DeviceID VARCHAR(20),
    IP_Address VARCHAR(50),
    MerchantID VARCHAR(20),
    Channel VARCHAR(20),
    CustomerAge INT,
    CustomerOccupation VARCHAR(50),
    TransactionDuration INT,
    LoginAttempts INT,
    AccountBalance FLOAT,
    PreviousTransactionDate DATETIME,

    time_since_last_txn FLOAT,
    amount_deviation_abs FLOAT,
    location_count_per_user INT,
    fraud_score INT,
    fraud_label VARCHAR(20)
);
"""

cursor.execute(create_table_query)
conn.commit()

print("Table created successfully")

Table created successfully


In [58]:
df = df.rename(columns={
    'IP Address': 'IP_Address'
})

In [60]:
df = df.rename(columns={
    'time_since_last_txn_fixed': 'time_since_last_txn'
})

In [67]:
print(len(columns))
print(len(data[0]))

21
22


In [68]:
print(len(df.columns))
print(df.columns)

29
Index(['TransactionID', 'AccountID', 'TransactionAmount', 'TransactionDate',
       'TransactionType', 'Location', 'DeviceID', 'IP_Address', 'MerchantID',
       'Channel', 'CustomerAge', 'CustomerOccupation', 'TransactionDuration',
       'LoginAttempts', 'AccountBalance', 'PreviousTransactionDate',
       'time_since_last_txn', 'prev_txn_date_fixed', 'time_since_last_txn',
       'high_velocity_flag', 'avg_txn_amount_per_user', 'amount_deviation_abs',
       'amount_anomaly_flag', 'location_count_per_user', 'location_risk_level',
       'fraud_score', 'fraud_label', 'anomaly_score', 'ml_flag'],
      dtype='object')


In [69]:
df = df[[
    'TransactionID','AccountID','TransactionAmount','TransactionDate',
    'TransactionType','Location','DeviceID','IP_Address','MerchantID',
    'Channel','CustomerAge','CustomerOccupation','TransactionDuration',
    'LoginAttempts','AccountBalance','PreviousTransactionDate',
    'time_since_last_txn','amount_deviation_abs',
    'location_count_per_user','fraud_score','fraud_label'
]]

In [70]:
print(len(df.columns))   # MUST be 21

22


In [71]:
df = df.loc[:, ~df.columns.duplicated()]

In [72]:
df = df[[
    'TransactionID','AccountID','TransactionAmount','TransactionDate',
    'TransactionType','Location','DeviceID','IP_Address','MerchantID',
    'Channel','CustomerAge','CustomerOccupation','TransactionDuration',
    'LoginAttempts','AccountBalance','PreviousTransactionDate',
    'time_since_last_txn','amount_deviation_abs',
    'location_count_per_user','fraud_score','fraud_label'
]]

In [73]:
print(len(df.columns))   # MUST be 21

21


In [74]:
print(len(data))

2512


In [77]:
data = []

for _, row in df.iterrows():
    data.append((
        row['TransactionID'],
        row['AccountID'],
        row['TransactionAmount'],
        str(row['TransactionDate']),
        row['TransactionType'],
        row['Location'],
        row['DeviceID'],
        row['IP_Address'],
        row['MerchantID'],
        row['Channel'],
        row['CustomerAge'],
        row['CustomerOccupation'],
        row['TransactionDuration'],
        row['LoginAttempts'],
        row['AccountBalance'],
        str(row['PreviousTransactionDate']),
        row['time_since_last_txn'],
        row['amount_deviation_abs'],
        row['location_count_per_user'],
        row['fraud_score'],
        row['fraud_label']
    ))

In [78]:
print(len(data[0]))   

21


In [79]:
cursor.execute("DELETE FROM transactions")
conn.commit()

for row in data:
    cursor.execute("""
    INSERT INTO transactions (
        TransactionID, AccountID, TransactionAmount, TransactionDate,
        TransactionType, Location, DeviceID, IP_Address, MerchantID,
        Channel, CustomerAge, CustomerOccupation, TransactionDuration,
        LoginAttempts, AccountBalance, PreviousTransactionDate,
        time_since_last_txn, amount_deviation_abs,
        location_count_per_user, fraud_score, fraud_label
    )
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    """, row)

conn.commit()

print("✅ Insert success")

✅ Insert success
